<a href="https://colab.research.google.com/github/LarsVoermans/master-thesis-pead/blob/main/XGB_boosting_regressiondata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#All imports
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor



In [ ]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving train_feature_engineered.parquet to train_feature_engineered.parquet
Saving val_feature_engineered.parquet to val_feature_engineered.parquet


In [ ]:
#Loading the dataset
train = pd.read_parquet("train_feature_engineered.parquet")
val = pd.read_parquet("val_feature_engineered.parquet")

In [ ]:
#making a return class
target = "Return"

X_train = train.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After"])
y_train = train["Return"]

X_val = val.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After"])
y_val = val["Return"]


In [ ]:
#Dropping columns that do not matter
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])


In [ ]:
# Making a k-fold
X_all = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_all = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

# K-Fold setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []
r2_scores = []
mae_scores = []
rmse_scores = []



In [ ]:
#Training the model
for train_idx, val_idx in kf.split(X_all):
    X_train_fold = X_all.iloc[train_idx]
    X_val_fold = X_all.iloc[val_idx]
    y_train_fold = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    # 3. XGBRegressor
    xgb_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,  # categorical columns worden ondersteund
        random_state=42
    )

    xgb_model.fit(X_train_fold, y_train_fold)
    y_pred = xgb_model.predict(X_val_fold)

    # 4. Metrics
    mse = mean_squared_error(y_val_fold, y_pred)
    r2 = r2_score(y_val_fold, y_pred)
    mae = mean_absolute_error(y_val_fold, y_pred)
    rmse = np.sqrt(mse)

    mse_scores.append(mse)
    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)


In [ ]:
# 5. Average score per fold
print("MSE per fold:", mse_scores)
print("Average MSE:", np.mean(mse_scores))
print("R² per fold:", r2_scores)
print("Average R²:", np.mean(r2_scores))
print("MAE per fold:", mae_scores)
print("Average MAE:", np.mean(mae_scores))
print("RMSE per fold:", rmse_scores)
print("Average RMSE:", np.mean(rmse_scores))

MSE per fold: [177.40290679321222, 149.98468245807376, 308.1726192674557, 158.19829479794524, 118.34137168838873]
Average MSE: 182.4199750010151
R² per fold: [-0.23268142361232402, -0.19539416609741966, -0.20298175517510164, -0.0003075129645673247, 0.13047995245239963]
Average R²: -0.10017698107940261
MAE per fold: [4.849236606123618, 4.4419699141184, 4.829217890011518, 4.4841634545404885, 4.438895920753412]
Average MAE: 4.608696757109487
RMSE per fold: [np.float64(13.31926825291886), np.float64(12.246823361920175), np.float64(17.554846033715467), np.float64(12.577690360234872), np.float64(10.878482048906857)]
Average RMSE: 13.315422011539246
